# Validierung: Einheiten, feste Verträge und echte OOF-Vorhersagen

Dieses Notebook liest die gespeicherten Zuordnungen und die echten Entwicklungs-OOF-Vorhersagen des Exponierungs-Basismodells. Es teilt keine Verträge neu auf und passt kein Modell an. Es lädt keine Ergebnisgrößen aus dem Evaluationsbestand. Vor dem ersten Lauf im Projektstamm `python -m src.validation freeze` und `python -m src.validation oof-baseline` ausführen.

Das vollständige statistische Protokoll steht in [docs/validierung.md](../docs/validierung.md).

In [ ]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "config/validation_protocol.json").is_file()
             and (p / "src/validation.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Notebook im Projektstamm oder einem Unterverzeichnis starten.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.validation import read_artifacts, identity_id
from src.metrics import poisson_deviance_per_exposure

split_public = json.loads((ROOT / "reports/s2/split_manifest.json").read_text(encoding="utf-8"))
run_public = json.loads((ROOT / "reports/s2/baseline_oof_report.json").read_text(encoding="utf-8"))
split_tables, split_report = read_artifacts(ROOT / "data/validation" / split_public["split_id"])
run_tables, run_report = read_artifacts(ROOT / "data/validation" / run_public["run_id"])
assert split_public == split_report
assert run_public == run_report
assert identity_id("s2split-", split_report["identity"]) == split_report["split_id"]
assert identity_id("s2oof-", run_report["identity"]) == run_report["run_id"]
assert run_report["split_id"] == split_report["split_id"]
print("Quelle:", run_report["source_id"])
print("Aufteilung:", split_report["split_id"])
print("OOF-Lauf:", run_report["run_id"])

## Feste Mitgliedschaft statt wiederholter Zufallsauswahl

Die Zuordnung verwendet ausschließlich die sortierten Vertragsschlüssel und die festgelegte Zufallsregel. 20 % der Verträge bilden die Evaluation; der Entwicklungsbestand hat fünf Falten. Alle Schadenzeilen übernehmen die Mitgliedschaft ihres Vertrags. Die folgende Tabelle zeigt nur die Größe der Aufteilung, keine Evaluationsschäden oder Evaluationsraten.

Die historische Analyse betrachtete den vollständigen Datenstand; auch Quellenwahl und Zielabgleich erfolgten retrospektiv. Diese neue Aufteilung ist daher keine externe und keine buchstäblich nie gesehene Validierung. Ab dieser Festlegung bleibt die Evaluation aus neuer Exploration und Auswahl ausgeschlossen.

In [ ]:
membership = split_tables["policy_membership"]
claim_membership = split_tables["claim_membership"]
assert membership["IDpol"].is_unique
assert claim_membership["claim_row_id"].is_unique
inherited = claim_membership.merge(membership, on="IDpol", validate="many_to_one", suffixes=("", "_policy"))
assert inherited["split"].eq(inherited["split_policy"]).all()
assert inherited["fold"].eq(inherited["fold_policy"]).all()
assert len(inherited) == len(claim_membership)
display(pd.DataFrame([
    {"Bestand": "Entwicklung" if name == "development" else "Evaluation", "Verträge": count}
    for name, count in split_report["membership_sizes"].items()
]))
display(pd.DataFrame([
    {"Entwicklungsfalte": fold, "Verträge": count}
    for fold, count in split_report["development_fold_sizes"].items()
]))
print("Alle Schadenzeilen haben die Mitgliedschaft ihres Vertrags übernommen.")

## Die Einheit bestimmt das Gewicht

Eine Jahresrate `r` ergibt mit Exponierung `e` die erwartete Periodenanzahl `mu = e*r`. Der primäre Score ist `sum d(N, mu) / sum e`, mit `d(n,mu) = 2*(n*log(n/mu)-n+mu)` und logarithmischem Term null bei `n=0`. Er entspricht der mit Exponierung gewichteten mittleren Deviance von `N/e` und `r`. Bereits erwartete Anzahlen erhalten kein zweites Exponierungsgewicht. Die Deviance ist kein Prozentwert.

Das kleine Rechenbeispiel prüft diese Gleichheit unabhängig von den Portfoliodaten.

In [ ]:
from sklearn.metrics import mean_poisson_deviance
n_example = np.array([1, 0])
e_example = np.array([0.1, 1.0])
r_example = np.array([1.0, 1.0])
count_score = poisson_deviance_per_exposure(n_example, r_example, e_example)
rate_score = mean_poisson_deviance(n_example / e_example, r_example, sample_weight=e_example)
assert np.isclose(count_score, rate_score, rtol=1e-12)
display(pd.DataFrame({"Berechnung": ["Anzahl-Deviance / Exponierung", "Exponierungsgewichtete Raten-Deviance"],
                      "Score": [count_score, rate_score]}))

## Tatsächliche OOF-Abdeckung im Entwicklungsbestand

Jeder Entwicklungsvertrag wird von genau dem Fit vorhergesagt, der seine Falte ausgelassen hat. Das aktuelle Modell verwendet nur die Trainingsrate `sum N / sum e`; es ist noch kein GLM-/GAM-/Boosting-Vergleich. Die Zuordnung erfolgt über `IDpol`, nicht über die Reihenfolge der Zeilen.

In [ ]:
predictions = run_tables["oof_predictions"]
dev_membership = membership.loc[membership["split"] == "development"]
assert predictions["IDpol"].is_unique
assert set(predictions["IDpol"]) == set(dev_membership["IDpol"])
assert predictions["split"].eq("development").all()
assert predictions["model_id"].eq(run_report["model_id"]).all()
assert predictions["run_id"].eq(run_report["run_id"]).all()
coverage = predictions.merge(dev_membership, on="IDpol", validate="one_to_one", suffixes=("", "_expected"))
assert coverage["fold"].eq(coverage["fold_expected"]).all()
assert run_report["evaluation_predictions"] == 0
assert run_report["evaluation_scores"] == 0
summary_labels = {
    "policies": "Entwicklungsverträge", "exposure_years": "Exponierung [Vertragsjahre]",
    "observed_claims": "Beobachtete positive Schadenzeilen", "expected_claims": "Erwartete Periodenanzahl",
    "observed_annual_rate": "Beobachtete Jahresrate", "predicted_annual_rate": "Vorhergesagte Jahresrate",
    "actual_over_predicted": "A/P: tatsächlich / vorhergesagt",
    "poisson_deviance_per_exposure": "Poisson-Deviance je Vertragsjahr",
    "mean_count_deviance": "Mittlere Anzahl-Deviance je Vertragszeile"
}
display(pd.DataFrame([
    {"Größe": summary_labels[name], "Entwicklungswert": value}
    for name, value in run_report["summary"].items()
]))
print("Entwicklungs-OOF-Zeilen:", len(predictions), "| Evaluationsvorhersagen: 0")

## Falten und korrekte Zusammenfassung

Die vier Trainingsfalten überlappen sich zwischen den Läufen. Ihre fünf Werte sind deshalb keine unabhängigen Wiederholungen für einen naiven Standardfehler. Der gepoolte Score ist mit Faltenexponierung gewichtet; ein ungewichtetes Mittel der fünf Quotienten wäre eine andere Größe.

In [ ]:
folds = pd.DataFrame(run_report["fold_diagnostics"])
weighted_fold_score = np.average(folds["deviance_per_exposure"], weights=folds["validation_exposure_years"])
assert np.isclose(weighted_fold_score, run_report["summary"]["poisson_deviance_per_exposure"], rtol=1e-12)
fold_columns = {
    "fold": "Falte", "training_policies": "Trainingsverträge", "validation_policies": "Validierungsverträge",
    "training_exposure_years": "Training [Vertragsjahre]", "training_claims": "Trainingsschäden",
    "validation_exposure_years": "Validierung [Vertragsjahre]", "validation_claims": "Validierungsschäden",
    "deviance_per_exposure": "Deviance je Vertragsjahr"
}
display(folds[list(fold_columns)].rename(columns=fold_columns))
print("Exponierungsgewichteter Faltenscore:", weighted_fold_score)

## Kalibrierung und rohe Frequenzkonzentration

Die Kalibrierungsgruppen ordnen Jahresraten aufsteigend und halten exakt gleiche Raten zusammen. Daher sind hier weniger als zehn Gruppen möglich. A/P hat die Richtung beobachtete durch erwartete Anzahl. Die Konzentrationskurve ordnet Jahresraten absteigend und verwendet kumulierte Exponierung und beobachtete Anzahlen, einschließlich Ursprung und ungeteilter Gleichstände.

Die fünf OOF-Intercepts haben trainingsspezifische Raten. Ihr gepoolter Gini kann von null abweichen; das belegt keine sinnvolle Risikodifferenzierung durch ein Interceptmodell.

In [ ]:
calibration = run_tables["calibration"]
assert calibration["policies"].sum() == len(predictions)
assert calibration["observed_claims"].sum() == run_report["summary"]["observed_claims"]
assert np.isclose(calibration["exposure_years"].sum(), run_report["summary"]["exposure_years"])
calibration_labels = {
    "group": "Gruppe", "min_annual_rate": "Kleinste Jahresrate", "max_annual_rate": "Größte Jahresrate",
    "policies": "Verträge", "exposure_years": "Exponierung [Vertragsjahre]",
    "observed_claims": "Beobachtete Anzahlen", "expected_claims": "Erwartete Anzahlen",
    "observed_annual_rate": "Beobachtete Jahresrate", "predicted_annual_rate": "Vorhergesagte Jahresrate",
    "actual_over_predicted": "A/P"
}
display(calibration.rename(columns=calibration_labels))
curve = run_tables["concentration"]
assert np.allclose(curve.iloc[0].to_numpy(), [0, 0])
assert np.allclose(curve.iloc[-1].to_numpy(), [1, 1])
raw_gini = 2 * np.trapezoid(curve["cumulative_claim_fraction"], curve["cumulative_exposure_fraction"]) - 1
display(curve.rename(columns={"cumulative_exposure_fraction": "Kumulierte Exponierung",
                             "cumulative_claim_fraction": "Kumulierte beobachtete Anzahlen"}))
print("Roher gepoolter OOF-Gini des Basismodells:", raw_gini)

## Was folgt und was noch nicht behauptet wird

Das Protokoll begrenzt den späteren Vergleich auf höchstens sechs GLM-, sechs GAM- und vier Histogramm-Boosting-Konfigurationen auf denselben Entwicklungsfalten. Konkrete Terme müssen aus der Entwicklungs-EDA begründet und vor der Anpassung registriert werden. Die 0,5-%-Toleranz für praktisch ähnliche Scores ist kein Signifikanztest.

Gepaarte Faltendifferenzen und 500 gepaarte Vertrags-Bootstrap-Ziehungen sind für den späteren Vergleich geplant. Intervalle bei festen OOF-Vorhersagen enthalten weder erneutes Fitten noch Modellwahlunsicherheit. Sie wurden hier nicht als Vergleichsergebnis berechnet. Auswahl anhand derselben CV-Werte ist keine unverzerrte Bewertung des abgestimmten Verfahrens.

Die Evaluation bleibt für neue Exploration und Auswahl geschlossen. Eine spätere Untersuchung synthetischer Trainingsdaten hat keine zusätzliche unabhängige Reserve und bleibt ohne neue Daten explorativ. Es liegen hier keine neuen adjustierten Alterseffekte und keine GLM-/GAM-/Boosting-Rangfolge vor.

In [ ]:
display(pd.DataFrame([
    {"Artefakt": name, "Zeilen": spec["rows"], "Datei": spec["filename"], "SHA-256": spec["sha256"]}
    for name, spec in run_report["artifacts"].items()
]))
print("Gespeicherte Entwicklungsartefakte und schlüsselbasierte OOF-Abdeckung geprüft.")